# Goodreads Books — Data Overview

Notebook này chỉ đọc dữ liệu raw, quan sát schema, xuất Parquet và thực hiện basic EDA. Không preprocessing sâu, feature engineering, embedding hay recommendation model.

In [9]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DATA_DIR = PROJECT_ROOT / "data/processed"
PARQUET_PATH = RAW_DATA_DIR / "goodreads_books.parquet"

PARQUET_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f"RAW_DATA_DIR: {RAW_DATA_DIR.resolve()}")
print(f"PARQUET_PATH: {PARQUET_PATH.resolve()}")

RAW_DATA_DIR: D:\Projects\BookRecSys\data\processed
PARQUET_PATH: D:\Projects\BookRecSys\data\raw_data\goodreads_books.parquet


## 1. Load và kiểm tra schema dataset gốc

In [10]:
from datasets import Dataset, DatasetDict, load_from_disk

raw_object = load_from_disk(str(RAW_DATA_DIR))
print("Object type:", type(raw_object))

if isinstance(raw_object, DatasetDict):
    print("DatasetDict splits:", list(raw_object.keys()))
    split_name = "train" if "train" in raw_object else next(iter(raw_object))
    dataset = raw_object[split_name]
    print("Selected split:", split_name)
elif isinstance(raw_object, Dataset):
    dataset = raw_object
    print("Dataset split: single dataset")
else:
    raise TypeError(f"Unsupported object type: {type(raw_object)}")

print("Samples:", len(dataset))
print("Columns/features:", dataset.column_names)
print("Schema:")
print(dataset.features)

Object type: <class 'datasets.arrow_dataset.Dataset'>
Dataset split: single dataset
Samples: 205000
Columns/features: ['title', 'contributors', 'first_author', 'description', 'genres', 'series', 'format', 'pages', 'language', 'asin', 'isbn10', 'isbn13', 'publisher', 'publication_date', 'average_rating', 'ratings_count', 'reviews_count', 'work_id', 'catalog_guideline_violation', 'book_id', 'url', 'num_editions', 'has_more_editions', 'is_part_of_series']
Schema:
{'title': Value('string'), 'contributors': List({'name': Value('string'), 'url': Value('string'), 'role': Value('string'), 'num_followers': Value('int64'), 'num_distinct_works': Value('int64')}), 'first_author': Value('string'), 'description': Value('string'), 'genres': List(Value('string')), 'series': List({'name': Value('string'), 'position': Value('string')}), 'format': Value('string'), 'pages': Value('int64'), 'language': Value('string'), 'asin': Value('string'), 'isbn10': Value('string'), 'isbn13': Value('string'), 'publishe

In [11]:
# Quan sát khoảng 5 records đầu tiên
for index, record in enumerate(dataset.select(range(min(5, len(dataset))))):
    print(f"\n--- Record {index + 1} ---")
    print(record)


--- Record 1 ---
{'title': 'La doctrina secreta. Vol I. Cosmogenesis (Ciencia Espiritual / Spiritual Sciences)', 'contributors': [{'name': 'Helena Petrovna Blavatsky', 'url': 'https://www.goodreads.com/author/show/18295027.Helena_Petrovna_Blavatsky', 'role': 'Author', 'num_followers': 710, 'num_distinct_works': 1678}], 'first_author': 'Helena Petrovna Blavatsky', 'description': 'Esta es una obra inteligente, original e instructiva. Sus alcances permiten entender con claridad cómo se puede convertir a la astrologia en instrumento de desarrollo espiritual y psicologico. En contraste con lo tradicionalmente predictivo, el autor se esmera aqui en apoyarse en la ley del karma y el impulso de transformacion, subrayando el significado transformador karmico de Saturno, Urano, Neptuno y Pluton. Asi es como este trabajo es adecuado para todos los que, interesados en la astrologia, reconocen en ella un significado espiritual y un valor que aumenta nuestro autoconocimeinto. El contenido de esta o

## 2. Export raw dataset sang Parquet

In [12]:
import json
import numpy as np

# Convert Hugging Face Dataset -> Pandas only for Parquet export.
# Serialize nested columns to JSON strings so Parquet keeps exactly 24 top-level columns
# and viewers do not expand them into *.list.element leaf fields.
df_export = dataset.to_pandas()

nested_cols = ["contributors", "genres", "series"]

def to_json_safe(value):
    if value is None:
        return None
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, np.generic):
        value = value.item()
    return value

for col in nested_cols:
    df_export[col] = df_export[col].apply(
        lambda x: json.dumps(to_json_safe(x), ensure_ascii=False)
        if x is not None
        else None
    )

df_export.to_parquet(str(PARQUET_PATH), index=False)

file_size_bytes = PARQUET_PATH.stat().st_size
print("Exported:", PARQUET_PATH.resolve())
print(f"Size: {file_size_bytes / 1024**2:.2f} MB ({file_size_bytes / 1024**3:.2f} GB)")
print("Parquet columns:", len(df_export.columns))


Exported: D:\Projects\BookRecSys\data\raw_data\goodreads_books.parquet
Size: 109.48 MB (0.11 GB)
Parquet columns: 24


## 3. Đọc Parquet bằng Pandas và basic overview

In [13]:
import json
import pandas as pd

df = pd.read_parquet(PARQUET_PATH)

# Restore nested columns from JSON strings back to Python list/dict objects.
nested_cols = ["contributors", "genres", "series"]
for col in nested_cols:
    df[col] = df[col].apply(
        lambda x: json.loads(x) if pd.notna(x) and x != "" else []
    )

display(df.shape)
display(df.head())
df.info()

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
display(pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str).values}))
print(f"DataFrame memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Round-trip validation for nested fields.
check_n = min(5, len(df))
for col in nested_cols:
    for i in range(check_n):
        assert dataset[i][col] == df.iloc[i][col], f"Mismatch at row={i}, column={col}"

print("ROUND-TRIP CHECK PASSED")


(205000, 24)

,title,contributors,first_author,description,genres,series,format,pages,language,asin,...,average_rating,ratings_count,reviews_count,work_id,catalog_guideline_violation,book_id,url,num_editions,has_more_editions,is_part_of_series
0,La doctrina secreta. Vol I. Cosmogenesis (Cien...,"[{'name': 'Helena Petrovna Blavatsky', 'url': ...",Helena Petrovna Blavatsky,"Esta es una obra inteligente, original e instr...","[Occult, Spirituality, Nonfiction, Philosophy,...","[{'name': 'The Secret Doctrine', 'position': '...",Paperback,319.0,Spanish; Castilian,950171103X,...,4.16,255.0,22.0,44066165.0,False,7282453,https://www.goodreads.com/book/show/7282453,124.0,True,True
1,Clement Marot and the Inflections of Poetic Voice,"[{'name': 'Robert Griffin', 'url': 'https://ww...",Robert Griffin,This title is part of UC Press's Voices Revive...,[],[],Hardcover,NaN,English,0520025865,...,0.00,0.0,0.0,1320719.0,False,1331224,https://www.goodreads.com/book/show/1331224,2.0,True,False
2,How to Market Legal Services,"[{'name': 'Robert W Denney', 'url': 'https://w...",Robert W Denney,NaN,[],[],Hardcover,278.0,English,0442219806,...,0.00,0.0,0.0,3956063.0,False,3910678,https://www.goodreads.com/book/show/3910678,2.0,True,False
3,Personality and Intellectual Competence,"[{'name': 'Tomas Chamorro-Premuzic', 'url': 'h...",Tomas Chamorro-Premuzic,This book provides a comprehensive state-of-th...,"[Psychology, Reference]",[],Hardcover,214.0,English,0805851364,...,3.88,8.0,0.0,580945.0,False,2027361,https://www.goodreads.com/book/show/2027361,8.0,True,False
4,Studies in the Text History Of the Life and Fa...,"[{'name': 'Ben Edwin Perry', 'url': 'https://w...",Ben Edwin Perry,"English, Greek",[],[],Paperback,272.0,English,0891305343,...,0.00,0.0,0.0,3997874.0,False,3952223,https://www.goodreads.com/book/show/3952223,4.0,True,False


<class 'pandas.DataFrame'>
RangeIndex: 205000 entries, 0 to 204999
Data columns (total 24 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   title                        205000 non-null  str    
 1   contributors                 205000 non-null  object 
 2   first_author                 204989 non-null  str    
 3   description                  178060 non-null  str    
 4   genres                       205000 non-null  object 
 5   series                       205000 non-null  object 
 6   format                       203038 non-null  str    
 7   pages                        189092 non-null  float64
 8   language                     197141 non-null  str    
 9   asin                         203887 non-null  str    
 10  isbn10                       193187 non-null  str    
 11  isbn13                       192592 non-null  str    
 12  publisher                    201887 non-null  str    
 13  publicatio

,column,dtype
0,title,str
1,contributors,object
2,first_author,str
3,description,str
4,genres,object
5,series,object
6,format,str
7,pages,float64
8,language,str
9,asin,str


DataFrame memory usage: 221.07 MB
ROUND-TRIP CHECK PASSED


## 4. Missing values và duplicate

In [14]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean().mul(100).round(2),
}).sort_values("missing_percent", ascending=False)
display(missing)

# Some columns contain lists/NumPy arrays, which are unhashable.
# Convert values to stable strings only for the duplicate check.
# Convert nested list/dict/array columns to strings only for duplicate detection.
duplicate_rows = int(df.astype(str).duplicated().sum())
print(f"Duplicate rows: {duplicate_rows:,}")
print(f"Duplicate rate: {duplicate_rows / len(df) * 100:.2f}%")

if "title" in df.columns:
    title_duplicate_count = int(df["title"].duplicated().sum())
    print(f"Unique titles: {df['title'].nunique(dropna=True):,}")
    print(f"Rows with duplicated title after first occurrence: {title_duplicate_count:,}")
else:
    print("No title column; title duplicate check skipped.")

,missing_count,missing_percent
description,26940,13.14
pages,15908,7.76
publication_date,12667,6.18
isbn13,12408,6.05
isbn10,11813,5.76
language,7859,3.83
publisher,3113,1.52
format,1962,0.96
asin,1113,0.54
first_author,11,0.01


Duplicate rows: 0
Duplicate rate: 0.00%
Unique titles: 198,203
Rows with duplicated title after first occurrence: 6,797


## 5. Các trường quan trọng cho Content-Based Recommendation

In [15]:
def print_examples(column, n=5):
    if column in df.columns:
        print(f"\n{column} examples:")
        display(df[column].head(n).to_frame())

if "title" in df.columns:
    print(f"Distinct titles: {df['title'].nunique(dropna=True):,}")
    print_examples("title")

if "description" in df.columns:
    description_length = df["description"].dropna().astype(str).str.len()
    print(f"Description missing: {df['description'].isna().sum():,}")
    display(description_length.agg(["min", "median", "mean", "max"]).to_frame("characters"))
else:
    print("No description column; description statistics skipped.")

tag_column = next((c for c in ["genres", "tags"] if c in df.columns), None)
if tag_column:
    print(f"{tag_column} missing: {df[tag_column].isna().sum():,}")
    print(f"{tag_column} non-missing examples (structure preserved, not exploded):")
    display(df[tag_column].head(5).to_frame())
else:
    print("No genres/tags column; skipped.")

rating_column = next((c for c in ["average_rating", "rating", "ratings"] if c in df.columns), None)
if rating_column:
    display(df[rating_column].describe()[["count", "mean", "50%", "min", "max"]].rename(index={"50%": "median"}).to_frame("value"))
else:
    print("No rating column; rating statistics skipped.")

author_column = next((c for c in ["first_author", "author", "authors"] if c in df.columns), None)
if author_column:
    print(f"Unique authors: {df[author_column].nunique(dropna=True):,}")
    display(df[author_column].value_counts(dropna=True).head(10).rename("book_count").to_frame())
else:
    print("No author column; author statistics skipped.")

Distinct titles: 198,203

title examples:


,title
0,La doctrina secreta. Vol I. Cosmogenesis (Cien...
1,Clement Marot and the Inflections of Poetic Voice
2,How to Market Legal Services
3,Personality and Intellectual Competence
4,Studies in the Text History Of the Life and Fa...


Description missing: 26,940


,characters
min,0.000000
median,489.000000
mean,619.672189
max,5342.000000


genres missing: 0
genres non-missing examples (structure preserved, not exploded):


,genres
0,"[Occult, Spirituality, Nonfiction, Philosophy,..."
1,[]
2,[]
3,"[Psychology, Reference]"
4,[]


,value
count,204998.000000
mean,2.700206
median,3.610000
min,0.000000
max,5.000000


Unique authors: 149,525


,book_count
first_author,
Unknown,780
Books LLC,446
Anonymous,422
Source Wikipedia,341
Various,283
William Shakespeare,220
Agatha Christie,152
Charles Dickens,139
Arthur Conan Doyle,102


## 6. Random samples để đọc trực tiếp

In [16]:
preferred_columns = ["title", "first_author", "author", "authors", "genres", "tags", "description", "average_rating", "rating"]
sample_columns = [column for column in preferred_columns if column in df.columns]
if sample_columns:
    display(df.sample(n=min(5, len(df)), random_state=42)[sample_columns])
else:
    display(df.sample(n=min(5, len(df)), random_state=42))

,title,first_author,genres,description,average_rating
61621,La valigetta del dottore,A.J. Cronin,"[Classics, Short Stories, Novels]",Nella Valigetta del dottore Cronin riprende il...,3.82
53656,"The Cambrian Fossils of Chengjiang, China: The...",Xian-guag Hou,"[Palaeontology, Science]",The Chengjiang biota is one of the most remark...,3.77
12855,Sweet Short Grass,Peter Inchbald,[],"Very good, unread, first edition, first impres...",0.00
118917,Managing NGOs in Developing Countries,Zafar Iqbal Qureshi,[],In this volume there are well-researched manag...,0.00
153740,Le Retour du Roi,J.R.R. Tolkien,"[Fantasy, Fiction, Classics, Adventure, Audiob...",The Return of the King is the third part of J....,4.59
